- Load and prepare the wildfire dataset 
- define chronological train/validation/test splits
- reserve 2020 as an independent stress-test year 
- verify dataset size and class balance.

In [17]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
 
warnings.filterwarnings("ignore")
 
RANDOM_STATE = 42
TARGET       = "next_day_risk_class"
OUT_DIR      = Path("../data/engineered")
PLOT_DIR     = Path("outputs")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
 
df = pd.read_parquet("../data/integrated/algeria_wildfire_dataset.parquet")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["commune_id", "date"]).reset_index(drop=True)
df["year"] = df["date"].dt.year
 
TRAIN_YEARS  = [2015, 2016, 2017, 2018]
VAL_YEARS    = [2019]
TEST_YEARS   = [2021, 2022, 2023, 2024, 2025]
STRESS_YEARS = [2020]
 
train_mask  = df["year"].isin(TRAIN_YEARS)
val_mask    = df["year"].isin(VAL_YEARS)
test_mask   = df["year"].isin(TEST_YEARS)
stress_mask = df["year"].isin(STRESS_YEARS)
 
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df.date.min().date()} -> {df.date.max().date()}")
print(f"Communes: {df.commune_id.nunique():,}")
for name, mask in [("TRAIN",train_mask),("VAL",val_mask),("TEST",test_mask),("STRESS-2020",stress_mask)]:
    pos = 100*(df.loc[mask, TARGET]>0).mean()
    print(f"  {name:<12} rows={mask.sum():>7,} | positive={pos:.1f}%")

Loaded: 156,480 rows x 42 columns
Date range: 2015-06-01 -> 2025-10-30
Communes: 1,245
  TRAIN        rows= 54,890 | positive=19.3%
  VAL          rows= 15,938 | positive=27.7%
  TEST         rows= 68,657 | positive=15.6%
  STRESS-2020  rows= 16,995 | positive=32.7%


## Drop Redundant Raw Features

- Justified by EDA correlation matrix:
  - **NDWI:** r=0.92 with NDVI, r=0.98 with NBR → fully redundant
  - **BUI:** r=0.99 with DMC → mathematically near-identical
  - **ISI:** r=0.92 with FWI → FWI already integrates ISI

- Other low-value columns:
  - **wind_dir, aspect_mean_deg:** no monotonic fire relationship
  - **wind_speed_kmh:** r=-0.006 with target, encoded in FWI already
  - **shrub_fraction, crop_fraction, grass_fraction:** burnable_fraction already aggregates them; keeping all adds noise without gain

In [18]:
DROP_COLS = [
    "NDWI", "BUI", "ISI",
    "wind_dir", "aspect_mean_deg", "wind_speed_kmh",
    "shrub_fraction", "crop_fraction", "grass_fraction",
]
drop_present = [c for c in DROP_COLS if c in df.columns]
df = df.drop(columns=drop_present)
print(f"Dropped {len(drop_present)}: {drop_present}")
print(f"Columns remaining: {df.shape[1]}")


Dropped 9: ['NDWI', 'BUI', 'ISI', 'wind_dir', 'aspect_mean_deg', 'wind_speed_kmh', 'shrub_fraction', 'crop_fraction', 'grass_fraction']
Columns remaining: 33


### Fire Persistence Features

- EDA showed: same-day fire_count r=0.185, mean_frp r=0.251 with target.
- The saturation curve (0 fires->16% risk, 1 fire->76%, 3+->90%+) means rolling history and intensity capture what daily counts alone cannot.
- frp_total = fire_count * mean_frp encodes BOTH occurrence and intensity in one term 
- Additional features capture recent fire history, days since the last fire, and fire activity in neighboring communes within the same wilaya.

In [19]:
df["fire_count"] = df["fire_count"].fillna(0)
df["mean_frp"]   = df["mean_frp"].fillna(0)
df["frp_total"]  = df["fire_count"] * df["mean_frp"]  # intensity interaction
df["fire_active"] = (df["fire_count"] > 0).astype("int8")
 
def rolling_sum(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=1).sum()
 
def rolling_mean(group, col, window):
    return group[col].shift(1).rolling(window, min_periods=1).mean()
 
grp = df.groupby("commune_id", group_keys=False)
 
df["fire_count_3d"]  = grp.apply(lambda g: rolling_sum(g,  "fire_count", 3)).values
df["fire_count_7d"]  = grp.apply(lambda g: rolling_sum(g,  "fire_count", 7)).values
df["frp_total_7d"]   = grp.apply(lambda g: rolling_sum(g,  "frp_total",  7)).values
 
# days_since_fire — fire drought per commune
def days_since(series):
    result, count = [], 0
    for val in series:
        count = 0 if val > 0 else count + 1
        result.append(count)
    return pd.Series(result, index=series.index)
 
df["days_since_fire"] = grp["fire_count"].transform(days_since)
 
# Wilaya spatial lag — neighbor communes burning today (self excluded)
wilaya_sum = (df.groupby(["wilaya_id","date"])["fire_count"]
              .sum().reset_index()
              .rename(columns={"fire_count":"_w_sum"}))
df = df.merge(wilaya_sum, on=["wilaya_id","date"], how="left")
df["wilaya_fire_excl_self"] = (df["_w_sum"] - df["fire_count"]).clip(lower=0)
df = df.drop(columns=["_w_sum"])
 
print("Fire persistence features:")
for col in ["fire_count_3d","fire_count_7d","frp_total_7d","days_since_fire","wilaya_fire_excl_self"]:
    print(f"  {col}: mean={df[col].mean():.2f}  max={df[col].max():.1f}")


Fire persistence features:
  fire_count_3d: mean=0.91  max=510.0
  fire_count_7d: mean=2.12  max=694.0
  frp_total_7d: mean=41.99  max=16033.1
  days_since_fire: mean=31.31  max=141.0
  wilaya_fire_excl_self: mean=2.46  max=2881.0


### Commune Historical Fire Rate

- Compute each commune's historical fire rate using **training data only** to avoid leakage.
- Assign the resulting `commune_fire_rate` to all observations from that commune.
- For communes with no training-period data, use the **global training positive rate** as the fallback.
- This captures the idea that communes with a higher historical fire frequency are intrinsically higher-risk locations.

In [20]:
train_fire_rate = (
    df[train_mask]
    .groupby("commune_id")[TARGET]
    .apply(lambda x: (x > 0).mean())
    .reset_index()
    .rename(columns={TARGET: "commune_fire_rate"})
)
global_rate = (df[train_mask][TARGET] > 0).mean()
df = df.merge(train_fire_rate, on="commune_id", how="left")
df["commune_fire_rate"] = df["commune_fire_rate"].fillna(global_rate)
 
print(f"commune_fire_rate: mean={df.commune_fire_rate.mean():.3f}  "
      f"min={df.commune_fire_rate.min():.3f}  max={df.commune_fire_rate.max():.3f}")
print(f"Global train positive rate (fill value): {global_rate:.3f}")


commune_fire_rate: mean=0.188  min=0.000  max=0.855
Global train positive rate (fill value): 0.193


### Drought Accumulation Features

- EDA class separation: DC HIGH-LOW = +118.87, DMC = +67.68 
- Rolling features capture the **recent trajectory and accumulated conditions**, not just today's values.
- `FWI_7d`, `DC_7d`, and `DMC_7d` capture sustained fire-weather and drought conditions over the previous 7 days.
- `precip_7d` captures accumulated rainfall as a suppression/moisture signal.
- `FWI_trend` measures whether conditions are worsening or recovering: positive = worsening, negative = recovering.

In [21]:
grp = df.groupby("commune_id", group_keys=False)
 
df["FWI_7d"]    = grp.apply(lambda g: rolling_mean(g, "FWI", 7)).values
df["DC_7d"]     = grp.apply(lambda g: rolling_mean(g, "DC",  7)).values
df["DMC_7d"]    = grp.apply(lambda g: rolling_mean(g, "DMC", 7)).values
df["precip_7d"] = grp.apply(lambda g: rolling_sum(g,  "precip_mm", 7)).values
df["FWI_trend"] = df["FWI"] - df["FWI_7d"]  # positive = worsening
 
print(f"FWI_7d mean:    {df.FWI_7d.mean():.1f}")
print(f"DC_7d mean:     {df.DC_7d.mean():.1f}")
print(f"precip_7d mean: {df.precip_7d.mean():.2f} mm")
print(f"FWI_trend mean: {df.FWI_trend.mean():.2f}")

FWI_7d mean:    29.4
DC_7d mean:     538.7
precip_7d mean: 7.97 mm
FWI_trend mean: 0.04


### Vegetation Anomaly Features

- Raw values encode absolute greenness per commune, so a dense forest can have high NDVI regardless of risk.
- Anomalies capture whether vegetation is **DRIER THAN NORMAL** for that commune at that time of year.
- `NBR_anomaly` and `FFMC_anomaly` are computed relative to **train-period commune/month baselines** to avoid leakage.
- Missing commune/month baselines are filled using the global training mean.

In [22]:
train_ref = df[train_mask].copy()
 
for source_col, anomaly_col in [("NBR","NBR_anomaly"), ("FFMC","FFMC_anomaly")]:
    baseline = (
        train_ref.groupby(["commune_id","month"])[source_col]
        .mean().reset_index().rename(columns={source_col: f"_{source_col}_base"})
    )
    df = df.merge(baseline, on=["commune_id","month"], how="left")
    # Fill missing baselines with global train mean
    global_base = train_ref[source_col].mean()
    df[f"_{source_col}_base"] = df[f"_{source_col}_base"].fillna(global_base)
    df[anomaly_col] = df[source_col] - df[f"_{source_col}_base"]
    df = df.drop(columns=[f"_{source_col}_base"])
 
print(f"NBR_anomaly:   mean={df.NBR_anomaly.mean():.4f}  std={df.NBR_anomaly.std():.4f}")
print(f"FFMC_anomaly:  mean={df.FFMC_anomaly.mean():.4f}  std={df.FFMC_anomaly.std():.4f}")
assert abs(df.loc[train_mask,"NBR_anomaly"].mean()) < 0.01, "Anomaly mean not ~0 on train"

NBR_anomaly:   mean=-0.0058  std=0.0502
FFMC_anomaly:  mean=1.4978  std=11.0505


### Temporal Cyclical Encoding

- Month and day-of-year are **cyclical**, so December is adjacent to January.
- Sin/cos encoding preserves this cyclic relationship without imposing ordinal assumptions.
- `month_sin/cos` captures seasonal patterns, while `doy_sin/cos` captures finer fire-season dynamics throughout the year.

In [23]:
doy = df["date"].dt.dayofyear
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"]   = np.sin(2 * np.pi * doy / 365.25)
df["doy_cos"]   = np.cos(2 * np.pi * doy / 365.25)
 
print("Temporal features: month_sin, month_cos, doy_sin, doy_cos")

Temporal features: month_sin, month_cos, doy_sin, doy_cos


### Final Feature Set Definition

- Define the final feature set based on:
  1. EDA Pearson correlation with the target
  2. EDA mutual information scores
  3. Domain knowledge (fire behavior triangle: fuel, weather, terrain)
  4. Anti-redundancy (one representative per correlated group)
  5. Leakage safety (all features observable before prediction day)

- The final features cover:
  - Same-day fire state
  - Fire persistence and spread
  - Historical commune fire propensity
  - Weather and FWI drought memory
  - Vegetation conditions and anomalies
  - Terrain
  - Land cover
  - Human exposure
  - Temporal seasonality

- Verify that all selected features exist in the dataset before modeling.

In [24]:
FINAL_FEATURES = [
    # Same-day fire state (strongest predictors per EDA)
    "fire_count", "mean_frp", "frp_total", "fire_active",
 
    # Fire history — persistence and spread
    "fire_count_3d", "fire_count_7d", "frp_total_7d",
    "days_since_fire", "wilaya_fire_excl_self",
 
    # Commune baseline fire propensity (train-period historical rate)
    "commune_fire_rate",
 
    # Weather (EDA: temp r=0.183, rh r=-0.119)
    "temp_c", "rh", "precip_mm", "soil_moisture",
 
    # FWI system — drought memory (EDA: DC separation +118.87, DMC +67.68)
    "FFMC", "DMC", "DC", "FWI",
    "FWI_7d", "DC_7d", "DMC_7d", "precip_7d", "FWI_trend",
 
    # Vegetation (real signal after type==0 filter: NDVI r=0.148, NBR r=0.136)
    "NDVI", "NBR", "NBR_anomaly", "FFMC_anomaly",
 
    # Terrain (EDA MI: slope 0.119, elevation 0.120)
    "elevation_mean_m", "slope_mean_deg",
 
    # Land cover
    "burnable_fraction", "forest_fraction",
 
    # Human exposure
    "pop_density_mean", "road_distance_mean_km",
 
    # Temporal encoding
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]
 
# Verify all features exist
missing_feats = [f for f in FINAL_FEATURES if f not in df.columns]
if missing_feats:
    print(f"❌ Missing features: {missing_feats}")
else:
    print(f"All {len(FINAL_FEATURES)} features present")
    print(f"Features: {FINAL_FEATURES}")

All 37 features present
Features: ['fire_count', 'mean_frp', 'frp_total', 'fire_active', 'fire_count_3d', 'fire_count_7d', 'frp_total_7d', 'days_since_fire', 'wilaya_fire_excl_self', 'commune_fire_rate', 'temp_c', 'rh', 'precip_mm', 'soil_moisture', 'FFMC', 'DMC', 'DC', 'FWI', 'FWI_7d', 'DC_7d', 'DMC_7d', 'precip_7d', 'FWI_trend', 'NDVI', 'NBR', 'NBR_anomaly', 'FFMC_anomaly', 'elevation_mean_m', 'slope_mean_deg', 'burnable_fraction', 'forest_fraction', 'pop_density_mean', 'road_distance_mean_km', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']


### Mutual Information Validation

- Compute mutual information scores using the **training split only** to prevent test-set information from influencing feature evaluation.
- Rank the final features by their MI score to verify that they carry useful signal for the target.
- Flag features with **MI < 0.003** as low-signal features.
- Low-MI features are retained when justified by **domain knowledge** and should be reviewed if model performance is poor.

In [25]:
X_mi = df.loc[train_mask, FINAL_FEATURES].fillna(0)
y_mi = df.loc[train_mask, TARGET]
 
mi_scores = mutual_info_classif(X_mi, y_mi, random_state=RANDOM_STATE)
mi_df = pd.DataFrame({"feature": FINAL_FEATURES, "mi_score": mi_scores})
mi_df = mi_df.sort_values("mi_score", ascending=False).reset_index(drop=True)
 
print("Mutual Information scores (train only):")
print(mi_df.to_string(index=False))
 
# Flag any feature with MI < 0.003 that we are force-keeping
low_mi = mi_df[mi_df["mi_score"] < 0.003]
if len(low_mi):
    print(f"\nLow MI features (<0.003): {low_mi.feature.tolist()}")
    print("   These are kept for domain reasons — review if model performance is poor.")
else:
    print("\nAll features have MI > 0.003")

Mutual Information scores (train only):
              feature  mi_score
     elevation_mean_m  0.127201
road_distance_mean_km  0.126196
       slope_mean_deg  0.125533
    commune_fire_rate  0.125508
     pop_density_mean  0.123211
    burnable_fraction  0.120513
          NBR_anomaly  0.114177
      forest_fraction  0.110283
                 NDVI  0.094324
                  NBR  0.087210
      days_since_fire  0.064494
         frp_total_7d  0.064254
             mean_frp  0.057376
            frp_total  0.055706
           fire_count  0.052902
          fire_active  0.051532
        fire_count_3d  0.046757
        fire_count_7d  0.044786
              doy_sin  0.041048
wilaya_fire_excl_self  0.040392
              doy_cos  0.040355
               temp_c  0.033606
        soil_moisture  0.032975
                   DC  0.030701
                  FWI  0.028105
                  DMC  0.027752
                 FFMC  0.023768
            month_sin  0.019844
            precip_7d  0.019224


### Missing Value Audit and Fill

- Audit the final raw feature sets for missing values.
- Handle missing rolling features caused by unavailable prior-day history.
- Use same-day values as a fallback where the corresponding base feature exists.
- Fill any remaining missing values with 0.
- Perform a final check to ensure no missing values remain before modeling.

In [26]:
missing = df[FINAL_FEATURES].isnull().sum()
missing = missing[missing > 0]

if len(missing):
    print("Missing values:")
    for col, n in missing.items():
        print(f"  {col}: {n:,} ({100*n/len(df):.1f}%)")

    # Rolling features: use same-day value when no prior history exists
    fallback_map = {
        "fire_count_3d": "fire_count",
        "fire_count_7d": "fire_count",
        "frp_total_7d": "frp_total",
        "FWI_7d": "FWI",
        "DC_7d": "DC",
        "DMC_7d": "DMC",
        "precip_7d": "precip_mm",
        "FWI_trend": "FWI",
    }

    for col, base in fallback_map.items():
        if col in missing.index:
            df[col] = df[col].fillna(df[base])

    # Remaining missing values
    df[FINAL_FEATURES] = df[FINAL_FEATURES].fillna(0)

# Final check
remaining = df[FINAL_FEATURES].isnull().sum().sum()
print(f"\nAfter fill: {remaining} missing values remaining")

assert remaining == 0, "Still has missing values"

Missing values:
  fire_count_3d: 1,245 (0.8%)
  fire_count_7d: 1,245 (0.8%)
  frp_total_7d: 1,245 (0.8%)
  FWI_7d: 1,245 (0.8%)
  DC_7d: 1,245 (0.8%)
  DMC_7d: 1,245 (0.8%)
  precip_7d: 1,245 (0.8%)
  FWI_trend: 1,245 (0.8%)

After fill: 0 missing values remaining


### Build Splits and Save

- Build the final modeling dataset using the selected features and target.
- Create chronologically separated train, validation, test, and 2020 stress-test splits.
- Verify class distributions and temporal separation between splits.
- Save the engineered dataset, individual splits, and feature metadata for modeling.

In [27]:
ID_COLS = ["date", "commune_id", "commune_name", "wilaya_id", "wilaya_name", "year"]
 
df_model     = df[ID_COLS + FINAL_FEATURES     + [TARGET]].copy()
 
train      = df_model[df_model["year"].isin(TRAIN_YEARS)].copy()
val        = df_model[df_model["year"].isin(VAL_YEARS)].copy()
test       = df_model[df_model["year"].isin(TEST_YEARS)].copy()
stress_20  = df_model[df_model["year"].isin(STRESS_YEARS)].copy()
 
for name, split in [("TRAIN",train),("VAL",val),("TEST",test),("STRESS-2020",stress_20)]:
    pos = 100*(split[TARGET]>0).mean()
    dist = split[TARGET].value_counts(normalize=True).sort_index().mul(100).round(1)
    print(f"{name:<12} rows={len(split):>7,}  positive={pos:.1f}%  {dict(dist)}")
 
assert set(TRAIN_YEARS).isdisjoint(VAL_YEARS)
assert set(TRAIN_YEARS).isdisjoint(TEST_YEARS)
assert set(VAL_YEARS).isdisjoint(TEST_YEARS)
assert set(STRESS_YEARS).isdisjoint(TEST_YEARS)
print("\nAll splits temporally disjoint")
 
# Save
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_model.to_parquet(OUT_DIR/"algeria_wildfire_engineered.parquet", index=False)
train.to_parquet(OUT_DIR/"train.parquet", index=False)
val.to_parquet(OUT_DIR/"val.parquet", index=False)
test.to_parquet(OUT_DIR/"test.parquet", index=False)
stress_20.to_parquet(OUT_DIR/"stress_2020.parquet", index=False)
 
feature_meta = {
    "target": TARGET,
    "final_features": FINAL_FEATURES,
    "id_cols": ID_COLS,
    "train_years": TRAIN_YEARS,
    "validation_years": VAL_YEARS,
    "test_years": TEST_YEARS,
    "stress_years": STRESS_YEARS,
    "n_features": len(FINAL_FEATURES),
    "n_train": len(train),
    "n_val": len(val),
    "n_test": len(test),
    "n_stress": len(stress_20),
}
with open(OUT_DIR/"feature_meta.json","w") as f:
    json.dump(feature_meta, f, indent=2)
 
print(f"\nSaved to {OUT_DIR}")
print(f"Features: {len(FINAL_FEATURES)}")


TRAIN        rows= 54,890  positive=19.3%  {0: np.float64(80.7), 1: np.float64(15.0), 2: np.float64(4.3)}
VAL          rows= 15,938  positive=27.7%  {0: np.float64(72.3), 1: np.float64(19.9), 2: np.float64(7.8)}
TEST         rows= 68,657  positive=15.6%  {0: np.float64(84.4), 1: np.float64(12.1), 2: np.float64(3.5)}
STRESS-2020  rows= 16,995  positive=32.7%  {0: np.float64(67.3), 1: np.float64(21.9), 2: np.float64(10.9)}

All splits temporally disjoint

Saved to ../data/engineered
Features: 37
